# 04 Early-Warning Label Creation RESET
Creates 6h, 12h and 24h warning labels and model-ready 12h dataset.



In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print('Python:', sys.executable)



In [ ]:
current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent if current_dir.name == 'notebooks' else current_dir
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_TABLES_DIR = PROJECT_ROOT / 'outputs' / 'tables'
OUTPUT_FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures'
for p in [PROCESSED_DATA_DIR, OUTPUT_TABLES_DIR, OUTPUT_FIGURES_DIR]: p.mkdir(parents=True, exist_ok=True)
features_path = PROCESSED_DATA_DIR / 'metropt3_features.csv'
labelled_path = PROCESSED_DATA_DIR / 'metropt3_labelled.csv'
model_ready_path = PROCESSED_DATA_DIR / 'metropt3_model_ready_12h.csv'



In [ ]:
if not features_path.exists(): raise FileNotFoundError('Run Step 3 first.')
df = pd.read_csv(features_path)
timestamp_col = 'timestamp' if 'timestamp' in df.columns else [c for c in ['Timestamp','time','Time','datetime','Datetime'] if c in df.columns][0]
df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors='coerce')
df = df.dropna(subset=[timestamp_col]).sort_values(timestamp_col).reset_index(drop=True)
if df[timestamp_col].max() < pd.Timestamp('2020-07-01'):
    raise ValueError('Feature dataset is partial. Rerun Steps 2 and 3.')
print('Input shape:', df.shape, 'End:', df[timestamp_col].max())



In [ ]:
failure_events = pd.DataFrame({
    'event_id':['F1','F2','F3','F4'],
    'failure_type':['Air leak','Air leak','Air leak','Air leak'],
    'failure_start':['2020-04-18 00:00','2020-05-29 23:30','2020-06-05 10:00','2020-07-15 14:30'],
    'failure_end':['2020-04-18 23:59','2020-05-30 06:00','2020-06-07 14:30','2020-07-15 19:00']
})
failure_events['failure_start'] = pd.to_datetime(failure_events['failure_start'])
failure_events['failure_end'] = pd.to_datetime(failure_events['failure_end'])
failure_events['inside_dataset_range'] = failure_events['failure_start'].ge(df[timestamp_col].min()) & failure_events['failure_end'].le(df[timestamp_col].max())
failure_events.to_csv(OUTPUT_TABLES_DIR / 'failure_events_for_label_creation.csv', index=False)
failure_events



In [ ]:
df_labelled = df.copy()
df_labelled['failure_interval'] = 0
df_labelled['failure_event_id'] = 'None'
for _, e in failure_events.iterrows():
    m = (df_labelled[timestamp_col] >= e.failure_start) & (df_labelled[timestamp_col] <= e.failure_end)
    df_labelled.loc[m, 'failure_interval'] = 1
    df_labelled.loc[m, 'failure_event_id'] = e.event_id
print('Failure interval rows:', int(df_labelled['failure_interval'].sum()))



In [ ]:
for hours in [6,12,24]:
    label = f'warning_{hours}h'; evcol = f'warning_{hours}h_event_id'
    df_labelled[label] = 0; df_labelled[evcol] = 'None'
    for _, e in failure_events.iterrows():
        m = (df_labelled[timestamp_col] >= e.failure_start - pd.Timedelta(hours=hours)) & (df_labelled[timestamp_col] < e.failure_start) & (df_labelled['failure_interval'] == 0)
        df_labelled.loc[m, label] = 1
        df_labelled.loc[m, evcol] = e.event_id
    print(label, int(df_labelled[label].sum()))
    print(df_labelled[evcol].value_counts().head(10))



In [ ]:
df_labelled['model_include'] = np.where(df_labelled['failure_interval'] == 1, 0, 1)
model_ready = df_labelled[df_labelled['model_include'] == 1].copy()
print('Model-ready shape:', model_ready.shape)



In [ ]:
records = []
for _, e in failure_events.iterrows():
    rec = {'event_id':e.event_id,'failure_start':e.failure_start,'failure_end':e.failure_end}
    for h in [6,12,24]:
        rec[f'warning_{h}h_start'] = e.failure_start - pd.Timedelta(hours=h)
        rec[f'warning_{h}h_rows'] = int((df_labelled[f'warning_{h}h_event_id'] == e.event_id).sum())
    records.append(rec)
pd.DataFrame(records).to_csv(OUTPUT_TABLES_DIR / 'failure_warning_windows.csv', index=False)



In [ ]:
class_records=[]
for h in [6,12,24]:
    label=f'warning_{h}h'; total=len(model_ready); warning=int(model_ready[label].sum()); normal=total-warning
    class_records.append({'label':label,'normal_count':normal,'warning_count':warning,'total_model_rows':total,'warning_percentage':warning/total*100,'imbalance_ratio_normal_to_warning':normal/warning if warning else np.nan})
class_dist=pd.DataFrame(class_records)
class_dist.to_csv(OUTPUT_TABLES_DIR / 'warning_label_class_distribution.csv', index=False)
plt.figure(figsize=(10,6)); x=np.arange(len(class_dist)); w=0.35; plt.bar(x-w/2,class_dist['normal_count'],w,label='Normal'); plt.bar(x+w/2,class_dist['warning_count'],w,label='Warning'); plt.xticks(x,class_dist['label']); plt.ylabel('Rows'); plt.title('Warning label class distribution'); plt.legend(); plt.tight_layout(); plt.savefig(OUTPUT_FIGURES_DIR/'warning_label_class_distribution.png', dpi=300); plt.show()
class_dist



In [ ]:
plt.figure(figsize=(12,4))
for i, e in failure_events.iterrows():
    y=i+1; ws=e.failure_start-pd.Timedelta(hours=12)
    plt.hlines(y, ws, e.failure_start, linewidth=8, label='12h warning' if i==0 else '')
    plt.hlines(y, e.failure_start, e.failure_end, linewidth=8, label='Failure' if i==0 else '')
    plt.text(e.failure_start, y+0.12, e.event_id)
plt.yticks(range(1,len(failure_events)+1), failure_events['event_id']); plt.xlabel('Time'); plt.title('12h warning windows and failures'); plt.legend(); plt.tight_layout(); plt.savefig(OUTPUT_FIGURES_DIR/'warning_windows_timeline_12h.png', dpi=300); plt.show()



In [ ]:
df_labelled.to_csv(labelled_path, index=False)
model_ready.to_csv(model_ready_path, index=False)
summary = pd.DataFrame({'item':['Input rows','Labelled rows','Model-ready rows','12h warning rows','Dataset end'], 'value':[df.shape[0],df_labelled.shape[0],model_ready.shape[0],int(df_labelled['warning_12h'].sum()),df_labelled[timestamp_col].max()]})
summary.to_csv(OUTPUT_TABLES_DIR / 'label_creation_summary.csv', index=False)
found = sorted([x for x in df_labelled['warning_12h_event_id'].unique() if x != 'None'])
checklist = pd.DataFrame({'task':['Full feature dataset loaded','F1-F4 warning events found','Labelled dataset saved','Model-ready dataset saved'], 'status':['Complete','Complete' if set(['F1','F2','F3','F4']).issubset(found) else f'Check: found {found}','Complete','Complete']})
checklist.to_csv(OUTPUT_TABLES_DIR / 'label_creation_completion_checklist.csv', index=False)
print('Saved:', labelled_path)
print('Saved:', model_ready_path)
summary
